<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Computer-Networks/08-network-security-and-privacy.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Computer Networks guideline](Computer-Networks.html)


## **Network Security and Privacy**

The previous chapter followed a browser from a URL to DNS, HTTP, caches, and content delivery. That request works only because many components accept information from networks they do not control. A resolver returns an address, routers choose a path, a server presents a key, and packets cross local and interdomain links. Network security asks when those claims should be trusted, what an adversary can observe or change, and how failures are contained.

Security is not a single encryption switch. Encryption can hide application bytes while IP addresses and timing remain visible. A valid certificate can authenticate a service while a compromised browser steals the plaintext. A firewall can block unsolicited connections while an authorized account abuses an API. The useful unit of reasoning is therefore an **asset under a stated threat model**, protected by several controls with different failure modes.

This chapter follows the same HTTPS request through four questions:

1. What must remain confidential, correct, authentic, available, and private?
2. Which attacker positions and capabilities are in scope?
3. Which cryptographic, routing, access, and operational controls enforce those goals?
4. What remains exposed when every intended control works?

::: {.callout-warning}
The code uses local cryptographic operations, synthetic packet metadata, and defensive state machines. It does not scan external systems, forge traffic, recover passwords, or provide exploitation steps. Security experiments should be limited to systems and networks for which explicit authorization has been granted.
:::

### **Security Goals and Threat Models**

#### **Confidentiality, Integrity, Authentication, and Availability**

**Confidentiality** restricts who can learn information. **Integrity** detects unauthorized modification. **Authentication** establishes a claim about a peer, message, device, or service. **Availability** keeps an authorized operation usable within an acceptable delay and capacity. These goals interact but are not substitutes. An encrypted response from an unauthenticated endpoint may be confidential to an attacker. A correctly authenticated server can still be unavailable under load.

![The CIA triad connects confidentiality, integrity, and availability.](assets/security-cia-triad.png){fig-alt="Circular information security diagram connecting confidentiality integrity and availability" width="56%"}

*Figure source: [Michel Bakni, CIA triad, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:CIAJMK1209-en.svg), licensed under CC BY-SA 4.0.*

Network systems usually add more precise goals:

| Goal | Question asked by a client | Typical control |
|---|---|---|
| service identity | Is this key authorized for `service.example`? | certificate path and service-identity validation |
| data origin | Which principal created this message? | signature or a channel authenticated to that principal |
| freshness | Is this message new rather than replayed? | nonce, sequence number, expiry, replay window |
| authorization | May this authenticated principal perform this action? | policy at the resource, not merely at login |
| accountability | Can an event be attributed and investigated? | protected, synchronized, retained logs |
| privacy | Can observers infer behavior from names, addresses, size, or timing? | data minimization, encryption, padding, relays |

#### **Assets, Adversaries, Trust Boundaries, and Attack Surfaces**

A useful threat model begins with **assets** such as credentials, private keys, session state, personal data, routing authority, availability budget, and audit evidence. It then names adversaries by capability: where can they observe traffic, can they inject packets, do they possess a valid account, or have they compromised an endpoint? A **trust boundary** is a point where data or authority crosses between principals with different control. Every parser, privileged API, certificate store, management interface, and externally reachable protocol contributes to the **attack surface**.

Risk is often ranked with a qualitative model such as

$$
R = L \times I \times E,
$$

where $L$ is likelihood, $I$ impact, and $E$ exposure. The score is a prioritization aid, not a measured probability. Its value comes from recording assumptions and comparing the same system consistently.

In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Threat:
    name: str
    asset: str
    likelihood: int          # ordinal scale: 1 (low) to 5 (high)
    impact: int              # ordinal scale: 1 (low) to 5 (high)
    exposure: int            # how broadly the vulnerable surface is exposed
    control_effectiveness: float  # estimated fractional reduction, not certainty

    @property
    def inherent_score(self):
        return self.likelihood * self.impact * self.exposure

    @property
    def residual_score(self):
        return self.inherent_score * (1 - self.control_effectiveness)


threats = [
    Threat("stolen session cookie", "account", 4, 5, 4, 0.70),
    Threat("DNS response injection", "service route", 3, 4, 3, 0.85),
    Threat("origin overload", "availability", 4, 5, 5, 0.55),
    Threat("compromised administrator", "control plane", 2, 5, 5, 0.35),
]

print("residual | inherent | threat -> asset")
for threat in sorted(threats, key=lambda item: item.residual_score, reverse=True):
    print(
        f"{threat.residual_score:8.1f} | {threat.inherent_score:8d} | "
        f"{threat.name} -> {threat.asset}"
    )

residual | inherent | threat -> asset
    45.0 |      100 | origin overload -> availability
    32.5 |       50 | compromised administrator -> control plane
    24.0 |       80 | stolen session cookie -> account
     5.4 |       36 | DNS response injection -> service route


The largest inherent score need not remain the largest residual risk. A strong control may reduce one path while a privileged or operational path remains weak. The numbers should trigger questions such as “What evidence supports 70%?” and “Which control failure was assumed?”, not create false precision.

#### **Passive and Active Attackers**

A **passive attacker** observes packets, headers, sizes, timing, radio transmissions, or routing updates but does not modify them. Encryption reduces payload disclosure, yet traffic analysis can still reveal destinations, periodic behavior, or large transfers. An **active attacker** can inject, alter, replay, redirect, delay, or drop traffic. Integrity protection and authentication address modification and impersonation; replay state and availability controls are still separate requirements.

#### **On-Path, Off-Path, Local, and Compromised-Endpoint Adversaries**

An **on-path** adversary sees traffic in both directions and can race or suppress legitimate messages. An **off-path** adversary normally cannot observe the flow and must guess values, exploit reflection, or influence a control plane. A **local** adversary may interact with ARP, Neighbor Discovery, Wi-Fi management frames, DHCP, and the configured resolver. A **compromised endpoint** is stronger than a network attacker: it can often read plaintext before encryption, access live credentials, and instruct the protocol to perform valid but malicious actions.

![Trust boundaries and attacker positions along one HTTPS request.](assets/security-threat-boundaries.svg){fig-alt="HTTPS request crosses browser access network Internet services CDN and origin with passive active and endpoint attacker positions" width="100%"}

The diagram exposes a crucial limit: TLS protects bytes between TLS endpoints. If a CDN terminates TLS, that edge is an endpoint for the connection and becomes part of the trusted service boundary. If the browser or origin is compromised, stronger ciphers do not restore application confidentiality.

### **Cryptographic Building Blocks for Networks**

Cryptographic protocols combine primitives; they do not obtain every property from one operation. A robust design authenticates a key exchange, binds every negotiated parameter into a transcript, derives independent directional keys, uses nonces correctly, and tracks replay state.

![Hash MAC signature authenticated encryption and key exchange provide different guarantees.](assets/network-crypto-toolbox.svg){fig-alt="Comparison table of hash MAC digital signature AEAD and key exchange with guarantees and limitations" width="100%"}

#### **Symmetric Encryption and Authenticated Encryption**

Symmetric encryption uses the same secret key family for encryption and decryption and is efficient enough for packet or record data. Modern network protocols normally use **authenticated encryption with associated data (AEAD)** rather than encryption alone:

$$
(C,T)=\operatorname{AEAD.Enc}(K,N,P,A),
$$

where $K$ is a traffic key, $N$ is a nonce, $P$ is plaintext, $A$ is associated data that remains visible but authenticated, $C$ is ciphertext, and $T$ is an authentication tag. Decryption returns plaintext only if the tag verifies. In TLS and QUIC, sequence numbers help construct unique nonces. Reusing a nonce with the same key can destroy security even when the algorithm is sound.

The example uses AES-GCM from a maintained cryptographic library. Its fixed key and nonce exist only to make notebook output reproducible; production code must generate keys securely and guarantee nonce uniqueness.

In [2]:
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# Deterministic teaching values. Never reuse a key/nonce pair in a real system.
key = bytes.fromhex("10" * 32)
nonce = bytes.fromhex("20" * 12)
associated_data = b"stream=client-to-server;sequence=7"
plaintext = b"POST /transfer amount=25"

aesgcm = AESGCM(key)
ciphertext_and_tag = aesgcm.encrypt(nonce, plaintext, associated_data)
recovered = aesgcm.decrypt(nonce, ciphertext_and_tag, associated_data)

print("plaintext bytes:", len(plaintext))
print("ciphertext + tag bytes:", len(ciphertext_and_tag))
print("valid record recovered:", recovered == plaintext)

tampered = bytearray(ciphertext_and_tag)
tampered[3] ^= 0x01
try:
    aesgcm.decrypt(nonce, bytes(tampered), associated_data)
except InvalidTag:
    print("tampered ciphertext: rejected before plaintext is released")

try:
    aesgcm.decrypt(nonce, ciphertext_and_tag, b"stream=server-to-client;sequence=7")
except InvalidTag:
    print("changed associated data: rejected")

plaintext bytes: 24
ciphertext + tag bytes: 40
valid record recovered: True
tampered ciphertext: rejected before plaintext is released
changed associated data: rejected


#### **Hash Functions and Message Authentication Codes**

A cryptographic hash maps arbitrary input to a fixed-length digest and should resist finding collisions or a preimage. Hashes summarize handshake transcripts, identify immutable objects, and build Merkle trees. A bare hash does **not** authenticate an untrusted message because an attacker can replace both message and digest.

A **message authentication code (MAC)** combines a shared secret with a message. HMAC is a standard construction based on a hash. A valid MAC shows that some holder of the shared key created the value and that it was not changed. It does not distinguish among several holders of the same key, and it does not hide plaintext.

#### **Public-Key Encryption and Digital Signatures**

Public-key systems separate public and private operations. Network protocols use public-key cryptography mainly to authenticate handshakes, certificates, routing objects, and signed updates. A digital signature is verified with a public key, but the verifier must still know **whose key it is** and **what context was signed**. Signing an ambiguous byte string can authenticate the wrong semantics.

#### **Key Exchange, Randomness, and Forward Secrecy**

Diffie-Hellman key exchange lets peers derive a shared secret over an observable channel. By itself it is vulnerable to an active man-in-the-middle: each side may establish a secret with the attacker. TLS authenticates the exchange by signing and MACing a transcript that includes the ephemeral shares and negotiated parameters.

![Diffie-Hellman allows two peers to derive the same secret without sending that secret.](assets/diffie-hellman-key-exchange.png){fig-alt="Alice and Bob combine private values with public Diffie Hellman values to obtain the same shared secret" width="45%"}

*Figure source: [A. J. Vinck and Flugaal, Diffie-Hellman Key Exchange, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Diffie-Hellman_Key_Exchange.svg), public domain.*

With ephemeral Diffie-Hellman, deleting the temporary private values provides **forward secrecy**: later compromise of the certificate private key should not decrypt previously recorded sessions. Secure randomness is therefore protocol state, not decoration. Weak keys, repeated nonces, predictable challenges, or VM snapshots that duplicate random state can invalidate otherwise correct algorithms.

In [3]:
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ed25519, x25519
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

# Deterministic private inputs keep output stable for this demonstration.
alice_private = x25519.X25519PrivateKey.from_private_bytes(bytes.fromhex("31" * 32))
bob_private = x25519.X25519PrivateKey.from_private_bytes(bytes.fromhex("52" * 32))

alice_shared = alice_private.exchange(bob_private.public_key())
bob_shared = bob_private.exchange(alice_private.public_key())
context = b"demo-handshake transcript hash"

def derive_traffic_key(shared_secret):
    return HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=context,
    ).derive(shared_secret)

alice_key = derive_traffic_key(alice_shared)
bob_key = derive_traffic_key(bob_shared)
print("same shared secret:", alice_shared == bob_shared)
print("same context-bound traffic key:", alice_key == bob_key)

# A server signs the transcript to authenticate the ephemeral exchange.
signing_key = ed25519.Ed25519PrivateKey.from_private_bytes(bytes.fromhex("73" * 32))
transcript = b"client_share|server_share|cipher_suite|service.example"
signature = signing_key.sign(transcript)
signing_key.public_key().verify(signature, transcript)
print("original transcript signature: valid")

try:
    signing_key.public_key().verify(signature, transcript + b"|downgraded")
except InvalidSignature:
    print("modified transcript signature: rejected")

same shared secret: True
same context-bound traffic key: True
original transcript signature: valid
modified transcript signature: rejected


A signature prevents transcript modification only if the public key is bound to the intended service. That identity binding is the job of certificates or a preconfigured key. The KDF also binds the shared secret to context, preventing one raw secret from being reused indiscriminately across protocols and directions.

**Freshness** still needs protocol state. A valid encrypted record copied from yesterday remains cryptographically valid unless a sequence number, nonce, timestamp, session identifier, or application idempotency rule rejects it. A sliding replay window accepts limited reordering while rejecting duplicates and records that are too old.

In [4]:
class ReplayWindow:
    """Fixed-width anti-replay window for monotonically numbered records."""

    def __init__(self, width=8):
        self.width = width
        self.highest = -1
        self.bitmap = 0

    def accept(self, sequence):
        if sequence < 0:
            return False, "invalid"

        if sequence > self.highest:
            shift = sequence - self.highest
            if shift >= self.width:
                self.bitmap = 0
            else:
                self.bitmap = (self.bitmap << shift) & ((1 << self.width) - 1)
            self.bitmap |= 1
            self.highest = sequence
            return True, "new highest"

        offset = self.highest - sequence
        if offset >= self.width:
            return False, "too old"

        bit = 1 << offset
        if self.bitmap & bit:
            return False, "duplicate"

        self.bitmap |= bit
        return True, "reordered but fresh"


window = ReplayWindow(width=8)
for sequence in [0, 1, 1, 4, 2, 12, 4, 11]:
    allowed, reason = window.accept(sequence)
    print(f"sequence={sequence:2d}: {'ACCEPT' if allowed else 'REJECT':6s} ({reason})")

sequence= 0: ACCEPT (new highest)
sequence= 1: ACCEPT (new highest)
sequence= 1: REJECT (duplicate)
sequence= 4: ACCEPT (new highest)
sequence= 2: ACCEPT (reordered but fresh)
sequence=12: ACCEPT (new highest)
sequence= 4: REJECT (too old)
sequence=11: ACCEPT (reordered but fresh)


### **Identity, Certificates, and Public-Key Infrastructure**

#### **Names, Identities, and Keys**

A DNS name, an account, a device identifier, and a public key are different objects. A key proves possession of a private value; it does not inherently say “this is `service.example`.” A certificate is a signed statement that binds identifiers and constraints to a public key. The client begins with a **reference identity** derived independently from the requested service, then checks whether the certificate presents an acceptable identifier. [RFC 9525](https://www.rfc-editor.org/rfc/rfc9525) defines current service-identity verification guidance and supersedes older RFC 6125 rules.

#### **X.509 Certificates and Certificate Authorities**

An X.509 certificate contains a subject public key, issuer, serial number, validity interval, signature, and extensions. Important extensions include Subject Alternative Name (SAN), Basic Constraints, Key Usage, Extended Key Usage, Name Constraints, and policies. A browser normally receives a leaf certificate and intermediates, then builds a path to a root already trusted by the local trust store.

![A leaf certificate is signed through intermediate authorities to a locally trusted root.](assets/certificate-chain-of-trust.png){fig-alt="Root certificate signs intermediate certificates which sign end entity certificates in a certificate chain of trust" width="78%"}

*Figure source: [Yanpas, Chain of trust, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Chain_of_trust.svg), licensed under CC BY-SA 4.0.*

Root trust is local policy. The root is usually self-signed, but its self-signature is not why it is trusted; the operating system or application distributed it as a trust anchor. Intermediates limit operational exposure and can carry path-length or name constraints.

#### **Certificate Validation and Revocation**

A simplified validation procedure is:

```text
INPUT: reference service identity, leaf, candidate intermediates, trust anchors, current time
1. Build a candidate path from leaf to a configured trust anchor.
2. Verify each issuer name and signature.
3. Check validity intervals, Basic Constraints, path length, and critical extensions.
4. Check Key Usage and Extended Key Usage for this protocol role.
5. Match the independently constructed service identity against SAN identifiers.
6. Apply revocation, certificate-transparency, algorithm, and local policy.
OUTPUT: one validated identity or a hard failure
```

The order matters conceptually even when a library implements it differently. Merely verifying the leaf signature skips path constraints and hostname identity; merely matching a hostname skips trust and expiry.

In [5]:
from datetime import datetime, timezone

from cryptography import x509
from cryptography.hazmat.primitives.asymmetric import ed25519
from cryptography.x509.oid import ExtendedKeyUsageOID, NameOID


def subject(common_name):
    return x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, common_name)])


not_before = datetime(2025, 1, 1, tzinfo=timezone.utc)
not_after_root = datetime(2035, 1, 1, tzinfo=timezone.utc)
not_after_intermediate = datetime(2032, 1, 1, tzinfo=timezone.utc)
not_after_leaf = datetime(2030, 1, 1, tzinfo=timezone.utc)

# Fixed private inputs make the demonstration reproducible.
root_key = ed25519.Ed25519PrivateKey.from_private_bytes(bytes.fromhex("81" * 32))
intermediate_key = ed25519.Ed25519PrivateKey.from_private_bytes(bytes.fromhex("82" * 32))
leaf_key = ed25519.Ed25519PrivateKey.from_private_bytes(bytes.fromhex("83" * 32))

root_name = subject("Blog Teaching Root")
root_cert = (
    x509.CertificateBuilder()
    .subject_name(root_name)
    .issuer_name(root_name)
    .public_key(root_key.public_key())
    .serial_number(1001)
    .not_valid_before(not_before)
    .not_valid_after(not_after_root)
    .add_extension(x509.BasicConstraints(ca=True, path_length=1), critical=True)
    .add_extension(
        x509.KeyUsage(False, False, False, False, False, True, True, False, False),
        critical=True,
    )
    .sign(root_key, algorithm=None)
)

intermediate_name = subject("Blog Teaching Intermediate")
intermediate_cert = (
    x509.CertificateBuilder()
    .subject_name(intermediate_name)
    .issuer_name(root_name)
    .public_key(intermediate_key.public_key())
    .serial_number(1002)
    .not_valid_before(not_before)
    .not_valid_after(not_after_intermediate)
    .add_extension(x509.BasicConstraints(ca=True, path_length=0), critical=True)
    .add_extension(
        x509.KeyUsage(False, False, False, False, False, True, True, False, False),
        critical=True,
    )
    .sign(root_key, algorithm=None)
)

leaf_name = subject("service.example")
leaf_cert = (
    x509.CertificateBuilder()
    .subject_name(leaf_name)
    .issuer_name(intermediate_name)
    .public_key(leaf_key.public_key())
    .serial_number(1003)
    .not_valid_before(not_before)
    .not_valid_after(not_after_leaf)
    .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
    .add_extension(
        x509.KeyUsage(True, False, False, False, False, False, False, False, False),
        critical=True,
    )
    .add_extension(
        x509.ExtendedKeyUsage([ExtendedKeyUsageOID.SERVER_AUTH]), critical=False
    )
    .add_extension(
        x509.SubjectAlternativeName(
            [x509.DNSName("service.example"), x509.DNSName("www.service.example")]
        ),
        critical=False,
    )
    .sign(intermediate_key, algorithm=None)
)


def verify_issued(child, issuer):
    assert child.issuer == issuer.subject
    issuer.public_key().verify(child.signature, child.tbs_certificate_bytes)


verify_issued(leaf_cert, intermediate_cert)
verify_issued(intermediate_cert, root_cert)

now = datetime(2026, 7, 20, tzinfo=timezone.utc)
names = leaf_cert.extensions.get_extension_for_class(
    x509.SubjectAlternativeName
).value.get_values_for_type(x509.DNSName)
eku = leaf_cert.extensions.get_extension_for_class(x509.ExtendedKeyUsage).value
intermediate_constraints = intermediate_cert.extensions.get_extension_for_class(
    x509.BasicConstraints
).value

print("path signatures: valid")
print("leaf time valid:", leaf_cert.not_valid_before_utc <= now <= leaf_cert.not_valid_after_utc)
print("service identity match:", "service.example" in names)
print("server authentication allowed:", ExtendedKeyUsageOID.SERVER_AUTH in eku)
print("intermediate is CA with path_length:", intermediate_constraints.path_length)
print("trust anchor selected locally:", root_cert.subject == root_name)

path signatures: valid
leaf time valid: True
service identity match: True
server authentication allowed: True
intermediate is CA with path_length: 0
trust anchor selected locally: True


The code verifies a teaching subset, not the complete [RFC 5280](https://www.rfc-editor.org/rfc/rfc5280) path algorithm. Production applications should use a platform verifier rather than recreate X.509 validation.

Revocation is operationally difficult. Certificate Revocation Lists can be large; OCSP introduces availability and privacy dependencies; stapling lets the server carry a signed status response but still has freshness and deployment constraints. A client must decide whether an unavailable status service causes a hard failure or a soft failure.

#### **Certificate Transparency and Trust-Store Risks**

**Certificate Transparency (CT)** makes certificate issuance publicly auditable through append-only Merkle-tree logs. It improves detection of misissuance; it does not prevent a CA from issuing the certificate or replace normal validation. [RFC 9162](https://www.rfc-editor.org/rfc/rfc9162) specifies CT version 2.0.

The public Web PKI distributes trust across many roots, intermediate operators, domain validation systems, logs, browser policies, clocks, and update channels. Reducing this risk requires CA constraints, short-lived credentials, key isolation, CT monitoring, automated rotation, and consistent configuration across every service endpoint. Pinning a key without a safe recovery path can turn normal rotation into an outage.

### **Transport Layer Security**

#### **TLS Handshake Goals**

TLS creates an authenticated, confidential channel over an untrusted path. A typical Web handshake must negotiate a supported protocol version and algorithms, derive fresh traffic secrets, authenticate the server for the requested identity, bind negotiation to one transcript, and confirm that both peers saw the same messages. TLS does not authorize an application action, hide destination IP addresses, or protect plaintext after an endpoint terminates the connection.

#### **TLS 1.3 Handshake and Key Schedule**

![TLS 1.3 negotiates key shares, authenticates the encrypted transcript, verifies Finished messages, and then derives application traffic keys.](assets/tls13-handshake-current.svg){fig-alt="TLS 1.3 full handshake sequence from ClientHello and ServerHello through Certificate CertificateVerify Finished and protected application data" width="92%"}

`ClientHello` and `ServerHello` select parameters and carry ephemeral key shares. Both sides derive handshake secrets, so later server authentication messages are encrypted. `CertificateVerify` signs the transcript with the server's certificate key. `Finished` authenticates the complete transcript with a derived secret. The client sends its own `Finished` after validating the path, service identity, signature, and server `Finished`.

TLS 1.3 uses HKDF to create a tree of secrets. At a high level,

$$
\text{secret}_{next}=\operatorname{HKDF\text{-}Extract}(\text{derived}_{current},\text{new input}),
$$

while `HKDF-Expand-Label` derives role- and context-specific traffic secrets. The transcript hash makes a key depend on the exact negotiation, and separate labels prevent a client write key from becoming a server write key.

In [6]:
import hashlib
import hmac

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDFExpand


def hkdf_extract(salt, input_key_material):
    return hmac.new(salt, input_key_material, hashlib.sha256).digest()


def hkdf_expand_label(secret, label, context, length=32):
    """TLS 1.3 HKDF-Expand-Label encoding from RFC 8446."""
    full_label = b"tls13 " + label
    info = (
        length.to_bytes(2, "big")
        + bytes([len(full_label)])
        + full_label
        + bytes([len(context)])
        + context
    )
    return HKDFExpand(
        algorithm=hashes.SHA256(), length=length, info=info
    ).derive(secret)


zero = bytes(32)
early_secret = hkdf_extract(zero, zero)  # no PSK in this full-handshake model
derived_early = hkdf_expand_label(early_secret, b"derived", hashlib.sha256(b"").digest())
handshake_secret = hkdf_extract(derived_early, alice_shared)

transcript_hash = hashlib.sha256(
    b"ClientHello|ServerHello|EncryptedExtensions|Certificate|CertificateVerify|Finished"
).digest()
client_handshake = hkdf_expand_label(handshake_secret, b"c hs traffic", transcript_hash)
server_handshake = hkdf_expand_label(handshake_secret, b"s hs traffic", transcript_hash)

derived_handshake = hkdf_expand_label(
    handshake_secret, b"derived", hashlib.sha256(b"").digest()
)
master_secret = hkdf_extract(derived_handshake, zero)
client_application = hkdf_expand_label(master_secret, b"c ap traffic", transcript_hash)
server_application = hkdf_expand_label(master_secret, b"s ap traffic", transcript_hash)

print("client/server handshake secrets differ:", client_handshake != server_handshake)
print("handshake/application secrets differ:", client_handshake != client_application)
print("client/server application secrets differ:", client_application != server_application)
print("client application secret prefix:", client_application.hex()[:16])

client/server handshake secrets differ: True
handshake/application secrets differ: True
client/server application secrets differ: True
client application secret prefix: 5256aa3334c947b5


#### **Record Protection and Connection Resumption**

The TLS record layer fragments application bytes and protects each record with AEAD. Directional sequence numbers contribute to nonce construction and integrity context. Sequence numbers are implicit in TLS over TCP because TCP delivers an ordered byte stream. QUIC protects packets and reconstructs packet numbers differently because loss and reordering are visible to QUIC.

A server can issue a resumption ticket that lets a later connection authenticate using a pre-shared key derived from the earlier session. Resumption reduces CPU and latency but creates ticket-key rotation and privacy responsibilities. Reusing a ticket can help a server correlate connections.

#### **Downgrade, Replay, and Man-in-the-Middle Defenses**

TLS 1.3 signs and MACs negotiated parameters, includes downgrade signals for compatibility paths, and removes many legacy algorithms. A man-in-the-middle cannot substitute key shares or certificate identities without failing transcript verification, assuming the client validates certificates correctly and its trust store is sound.

**0-RTT early data** is different: a resumed client can send encrypted application data before receiving fresh server proof, so the server or a distributed cluster may receive a replay. Only operations with an explicit replay-safe specification and application defense should use it. `GET` being nominally safe is not enough if the endpoint changes state, consumes a one-time token, or triggers an expensive action. [RFC 9325](https://www.rfc-editor.org/rfc/rfc9325) provides current TLS deployment guidance.

#### **TLS in TCP and QUIC**

HTTPS over TCP places TLS above TCP. Connection establishment normally costs a TCP handshake followed by the TLS exchange. QUIC integrates the TLS 1.3 handshake with transport setup and encrypts most transport metadata, but TLS records are not placed inside QUIC. QUIC derives packet-protection keys from TLS and maintains encryption levels for Initial, Handshake, 0-RTT, and 1-RTT packets. [RFC 9001](https://www.rfc-editor.org/rfc/rfc9001) defines this integration.

Neither substrate changes certificate identity rules. QUIC also requires anti-amplification limits before a client's address is validated, connecting transport security to availability.

### **Network-Layer Security and Virtual Private Networks**

#### **IPsec Authentication and Encapsulating Security Payload**

IPsec protects IP traffic using Security Associations (SAs), unidirectional state that identifies algorithms, keys, sequence state, and selectors. The Authentication Header (AH) authenticates selected packet fields but does not encrypt payload and interacts poorly with address translation. Encapsulating Security Payload (ESP) can provide confidentiality, integrity, origin authentication, and anti-replay for the protected portion and is the common choice.

IKEv2 authenticates peers and negotiates SAs, keys, lifetimes, and algorithms. Separating IKE from ESP mirrors the broader pattern: an expensive authenticated key-management protocol establishes efficient symmetric traffic protection.

#### **Transport and Tunnel Modes**

![IPsec ESP transport mode protects an IP payload while tunnel mode encapsulates the complete inner packet.](assets/ipsec-esp-modes.png){fig-alt="Comparison of IPsec ESP tunnel mode and transport mode packet layouts" width="90%"}

*Figure source: [Mpk1024, IPsec ESP tunnel and transport modes, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Ipsec-esp-tunnel-and-transport.svg), licensed under CC BY-SA 4.0.*

In **transport mode**, the original IP header remains the routing header and ESP protects the upper-layer payload. In **tunnel mode**, a new outer IP header carries an encrypted inner IP packet. Tunnel mode naturally connects gateways or hides internal addressing. What remains visible is determined by the outer packet: tunnel endpoints, total length, timing, and traffic direction remain observable.

#### **Site-to-Site and Remote-Access VPNs**

A site-to-site VPN connects network prefixes through gateways. A remote-access VPN gives one endpoint routes or policy access into an organization. The design must specify which prefixes enter the tunnel, how DNS is selected, whether Internet traffic is split or full tunnel, how routes are authorized, and what happens when the tunnel fails. “Connected” should not imply broad authorization to every internal service.

#### **Modern Encrypted Tunnels**

Modern tunnels such as WireGuard reduce protocol surface by fixing a compact cryptographic design and mapping peers to public keys and allowed IP prefixes. Simplicity does not remove operational work: private-key storage, peer provisioning, revocation, address overlap, MTU, keepalives, endpoint roaming, and access policy remain system concerns. A tunnel shifts the trusted observer from the local path to the tunnel endpoint; it is not automatic anonymity.

In [7]:
import math


def tunnel_payload_budget(
    outer_mtu,
    outer_ip=20,
    udp_encapsulation=8,
    tunnel_header=32,
    authentication_tag=16,
    inner_ip=20,
    tcp_header=20,
):
    """Illustrative budget; real ESP/tunnel overhead depends on negotiated details."""
    tunnel_overhead = outer_ip + udp_encapsulation + tunnel_header + authentication_tag
    inner_packet_mtu = outer_mtu - tunnel_overhead
    tcp_payload = inner_packet_mtu - inner_ip - tcp_header
    return tunnel_overhead, inner_packet_mtu, tcp_payload


for outer_mtu in (1500, 1280):
    overhead, inner_mtu, max_payload = tunnel_payload_budget(outer_mtu)
    segments = math.ceil(5000 / max_payload)
    print(
        f"outer MTU={outer_mtu}: overhead={overhead}, inner MTU={inner_mtu}, "
        f"max TCP payload={max_payload}, segments for 5000 bytes={segments}"
    )

outer MTU=1500: overhead=76, inner MTU=1424, max TCP payload=1384, segments for 5000 bytes=4
outer MTU=1280: overhead=76, inner MTU=1204, max TCP payload=1164, segments for 5000 bytes=5


Encapsulation reduces the inner packet budget. If endpoints keep sending a 1500-byte inner packet into a tunnel with a 1500-byte outer MTU, the result may be fragmentation or loss when Path MTU Discovery fails. Operators commonly adjust interface MTU or TCP MSS, then verify with packet evidence rather than guessing a universal overhead.

### **Network Access Control and Traffic Inspection**

#### **Packet Filters and Stateful Firewalls**

A stateless packet filter evaluates fields present in one packet: direction, interface, addresses, protocol, ports, and flags. It is fast and predictable but cannot know whether an inbound packet belongs to a connection initiated from inside. A **stateful firewall** tracks flow state and allows return traffic that matches an established or related exchange. State improves policy expression but consumes memory and creates timeout, failover, and exhaustion concerns.

#### **Access-Control Lists and Security Groups**

Router ACLs, host firewalls, cloud security groups, and Kubernetes network policies enforce reachability at different points. Good policy is narrow in source, destination, protocol, port, direction, and ownership, with an explicit default. An IP address is a routing locator, not a durable human or workload identity. Address-based controls remain useful barriers but should not be the only authorization for sensitive resources.

In [8]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Packet:
    direction: str  # "out" or "in" relative to the protected network
    protocol: str
    source: str
    source_port: int
    destination: str
    destination_port: int
    ack: bool = False


class TeachingStatefulFirewall:
    def __init__(self):
        self.expected_returns = set()

    def inspect(self, packet):
        flow = (
            packet.protocol,
            packet.source,
            packet.source_port,
            packet.destination,
            packet.destination_port,
        )

        if packet.direction == "out" and packet.destination_port in {53, 443}:
            reverse = (
                packet.protocol,
                packet.destination,
                packet.destination_port,
                packet.source,
                packet.source_port,
            )
            self.expected_returns.add(reverse)
            return "ALLOW: permitted outbound flow; return state created"

        if packet.direction == "in" and flow in self.expected_returns:
            return "ALLOW: packet matches return state"

        return "DROP: no policy or established return state"


firewall = TeachingStatefulFirewall()
packets = [
    Packet("in", "TCP", "198.51.100.9", 50000, "10.0.0.5", 22),
    Packet("out", "TCP", "10.0.0.5", 53000, "203.0.113.20", 443),
    Packet("in", "TCP", "203.0.113.20", 443, "10.0.0.5", 53000, ack=True),
    Packet("in", "TCP", "203.0.113.99", 443, "10.0.0.5", 53000, ack=True),
]
for packet in packets:
    print(packet, "\n ->", firewall.inspect(packet))

Packet(direction='in', protocol='TCP', source='198.51.100.9', source_port=50000, destination='10.0.0.5', destination_port=22, ack=False) 
 -> DROP: no policy or established return state
Packet(direction='out', protocol='TCP', source='10.0.0.5', source_port=53000, destination='203.0.113.20', destination_port=443, ack=False) 
 -> ALLOW: permitted outbound flow; return state created
Packet(direction='in', protocol='TCP', source='203.0.113.20', source_port=443, destination='10.0.0.5', destination_port=53000, ack=True) 
 -> ALLOW: packet matches return state
Packet(direction='in', protocol='TCP', source='203.0.113.99', source_port=443, destination='10.0.0.5', destination_port=53000, ack=True) 
 -> DROP: no policy or established return state


This state machine illustrates the difference between “source port 443” and “return packet for the exact flow.” Production connection tracking also handles TCP states, retransmission, NAT, ICMP errors, fragments, timeouts, and distributed failover.

#### **Intrusion Detection and Prevention**

An IDS observes and alerts; an IPS can block or modify traffic inline. Detection may use signatures, protocol invariants, reputation, behavior, or correlation. Inline prevention offers fast containment but false positives can become outages. Encrypted traffic limits payload inspection, so flow metadata, TLS fingerprints, endpoint telemetry, DNS logs, identity events, and application traces must be correlated.

#### **Proxies, Gateways, and Zero-Trust Segmentation**

A proxy terminates one connection and creates another, enabling application-aware authentication, policy, logging, normalization, and rate limiting. It also becomes a trusted endpoint that can see plaintext and hold credentials. API gateways and service meshes move policy closer to resources, but policy consistency and bypass paths matter more than product labels.

**Zero trust** removes implicit trust based only on network location. It does not mean “trust nothing” or “remove every firewall.” It means every access decision is explicit, least-privilege, resource-specific, and informed by subject, device, workload, and context. [NIST SP 800-207](https://csrc.nist.gov/pubs/sp/800/207/final) provides the architectural model.

![A zero-trust policy path evaluates identity device posture resource and context for every request.](assets/zero-trust-policy-path.svg){fig-alt="Subject and device pass through policy enforcement while a policy engine evaluates identity posture resource and risk" width="96%"}

### **Attacks Across the Protocol Stack**

A useful attack taxonomy connects each technique to a violated assumption and a defensive invariant. “Use TLS” is not an answer to a source-spoofed amplification attack; “install a firewall” is not an answer to a misissued certificate accepted by the browser.

![Cross-layer attacks violate different identity freshness capacity and authorization assumptions.](assets/cross-layer-attacks-defenses.svg){fig-alt="Table connecting attacks at application naming transport network link and endpoint layers to broken assumptions and controls" width="100%"}

#### **Eavesdropping, Spoofing, and Replay**

Eavesdropping observes communication. Spoofing presents a false source or identity. Replay resends a previously valid message. Their defenses differ: encryption reduces disclosure; cryptographic authentication detects forged records; source-address validation limits forged IP sources; sequence windows and application idempotency reject replays. A signed request without a nonce can be replayed, and an unpredictable source port does not authenticate a service.

#### **ARP and Neighbor-Discovery Attacks**

ARP and IPv6 Neighbor Discovery establish local link-layer next hops. Their basic forms assume a cooperative LAN, so a local attacker may advertise a false mapping or router. Mitigations include smaller trust domains, switch port security, DHCP snooping with dynamic ARP inspection, IPv6 RA Guard, authenticated access, static bindings for narrow cases, and end-to-end TLS. These controls must be validated against the actual switch and IPv6 behavior; one protected protocol does not automatically protect the other.

#### **SYN Floods and Transport Exhaustion**

A TCP server normally allocates partial state after a SYN. A flood can fill that queue before handshakes complete. SYN cookies or SYN proxies defer expensive state until a client demonstrates return-path reachability. Per-source limits, backlog tuning, short timeouts, anycast distribution, and upstream filtering add capacity barriers. QUIC uses address-validation tokens and an anti-amplification limit for a related reason.

#### **DNS Cache Poisoning and Amplification**

Cache poisoning attempts to insert a false DNS answer. Random transaction identifiers and source ports make blind guessing harder; bailiwick checks constrain accepted additional data; DNSSEC validates signed DNS data when a trust chain exists. DNSSEC supplies origin authentication and integrity, not query confidentiality.

Amplification uses a small request with a spoofed victim source to trigger a larger response from an open service. Its approximate factor is

$$
A=\frac{\text{response bytes}}{\text{request bytes}},
$$

but total harm also depends on reflectors, query rate, and upstream capacity. Source-address validation prevents a network from emitting spoofed traffic; response-rate limiting, resolver access control, anycast, and scrubbing reduce other parts of the path.

In [9]:
def reflected_load_gbps(
    request_bytes,
    response_bytes,
    reflectors,
    requests_per_second_each,
    spoofable_fraction=1.0,
    response_rate_limit_fraction=1.0,
):
    amplification = response_bytes / request_bytes
    responses_per_second = (
        reflectors
        * requests_per_second_each
        * spoofable_fraction
        * response_rate_limit_fraction
    )
    load_gbps = responses_per_second * response_bytes * 8 / 1_000_000_000
    return amplification, load_gbps


scenarios = {
    "unmitigated": (1.0, 1.0),
    "80% source validation": (0.2, 1.0),
    "source validation + response limits": (0.2, 0.1),
}

for name, (spoofable, rate_limit) in scenarios.items():
    factor, load = reflected_load_gbps(
        request_bytes=60,
        response_bytes=3000,
        reflectors=20_000,
        requests_per_second_each=40,
        spoofable_fraction=spoofable,
        response_rate_limit_fraction=rate_limit,
    )
    print(f"{name:36s}: amplification={factor:4.0f}x, victim load={load:6.2f} Gbit/s")

unmitigated                         : amplification=  50x, victim load= 19.20 Gbit/s
80% source validation               : amplification=  50x, victim load=  3.84 Gbit/s
source validation + response limits : amplification=  50x, victim load=  0.38 Gbit/s


#### **Denial of Service and Distributed Denial of Service**

Availability attacks target different bottlenecks: link bandwidth, packets per second, handshake state, CPU, memory, database work, dependency capacity, or human response. A defense must be placed before the constrained resource. Application rate limiting after a saturated access link cannot recover the link. Capacity planning should distinguish ordinary peaks, abusive but authenticated workloads, and spoofed or distributed floods.

#### **Web and Application-Layer Abuse at the Network Boundary**

At the application boundary, syntactically valid requests can consume disproportionate work, enumerate accounts, reuse leaked credentials, bypass object authorization, or exploit parsers. Controls include strict message limits, deadlines, bounded queues, authentication throttles, per-tenant quotas, object-level authorization, safe retry semantics, and dependency budgets. A web application firewall may add signatures and normalization, but it cannot infer correct business authorization without application context.

### **Routing Security**

#### **BGP Hijacks and Route Leaks**

A **BGP hijack** causes traffic to prefer an unauthorized origin or path, intentionally or accidentally. A **route leak** propagates a route beyond its intended commercial or topological scope, even when the origin is legitimate. Longest-prefix matching makes a more-specific false announcement especially influential. Consequences include blackholing, interception, asymmetric paths, certificate-validation disruption, and loss of reachability.

#### **RPKI and Route Origin Validation**

The Resource Public Key Infrastructure binds IP address resources and AS numbers to signed objects. A Route Origin Authorization (ROA) authorizes an AS to originate a prefix up to a maximum length. Validators produce validated ROA payloads (VRPs); routers compare received routes to those payloads and classify route origin state as **Valid**, **Invalid**, or **Not Found**.

![RPKI route origin validation compares a received prefix and origin AS with validated ROA payloads before local import policy.](assets/rpki-rov-decision.svg){fig-alt="BGP announcement and ROA enter a validator which returns valid invalid or not found before local policy" width="96%"}

[RFC 9582](https://www.rfc-editor.org/rfc/rfc9582) defines the current ROA profile. Validation state is an input to local policy, not a replacement for BGP route selection. A common design rejects Invalid routes while monitoring and assigning policy to Valid and Not Found routes.

In [10]:
import ipaddress
from dataclasses import dataclass


@dataclass(frozen=True)
class VRP:
    prefix: str
    max_length: int
    origin_as: int


def route_origin_state(route_prefix, route_origin_as, vrps):
    route = ipaddress.ip_network(route_prefix)
    covering = []
    for vrp in vrps:
        authorized = ipaddress.ip_network(vrp.prefix)
        if route.version == authorized.version and route.subnet_of(authorized):
            covering.append(vrp)

    if not covering:
        return "NOT FOUND"

    if any(
        route.prefixlen <= vrp.max_length and route_origin_as == vrp.origin_as
        for vrp in covering
    ):
        return "VALID"

    return "INVALID"


validated_payloads = [
    VRP("203.0.112.0/23", 24, 64520),
    VRP("2001:db8:1200::/40", 48, 64530),
]

routes = [
    ("203.0.113.0/24", 64520),  # covered, permitted length and ASN
    ("203.0.113.0/24", 64599),  # covered, wrong origin
    ("203.0.113.0/25", 64520),  # covered, too specific for maxLength
    ("198.51.100.0/24", 64550), # no covering VRP
]

for prefix, origin in routes:
    print(f"{prefix:18s} origin AS{origin}: {route_origin_state(prefix, origin, validated_payloads)}")

203.0.113.0/24     origin AS64520: VALID
203.0.113.0/24     origin AS64599: INVALID
203.0.113.0/25     origin AS64520: INVALID
198.51.100.0/24    origin AS64550: NOT FOUND


#### **Path Validation and Deployment Constraints**

ROV validates only the origin authorization. It does not prove that every AS in `AS_PATH` really propagated the route, prevent a valid origin from leaking a route, or guarantee that traffic follows the advertised reverse path. Operators still need route filters, prefix and AS-path limits, peer roles, monitoring from diverse vantage points, secure RPKI repositories and validators, staged policy changes, and incident contacts.

Stronger path mechanisms face incremental deployment, policy privacy, hardware, coordination, and failure-mode constraints. Security must improve routing without turning a repository or validator outage into global route withdrawal. Cached validated data, multiple validators, observability, and conservative change control are part of the security design.

### **Wireless Security**

#### **WPA2, WPA3, and Enterprise Authentication**

Wi-Fi security protects the radio link between a station and access point; end-to-end TLS remains necessary beyond that hop. In WPA2-Personal, a shared passphrase leads to a Pairwise Master Key (PMK). The four-way handshake proves both peers possess key material and derives fresh per-session keys using MAC addresses and nonces. The passphrase itself is not transmitted in the handshake.

![The WPA2 four-way handshake exchanges nonces and integrity proofs to install fresh traffic keys.](assets/wpa2-four-way-handshake.png){fig-alt="WPA2 802.11i four way handshake between supplicant and authenticator showing ANonce SNonce MIC and group key" width="62%"}

*Figure source: [Mikm, 802.11i four-way handshake, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:4-way-handshake.svg), public domain.*

WPA3-Personal uses Simultaneous Authentication of Equals (SAE), a password-authenticated key exchange that improves resistance to passive offline password guessing and supplies forward secrecy for the authentication exchange. WPA-Enterprise uses 802.1X/EAP with an authentication server and per-user or per-device credentials. Correct certificate validation by the supplicant is crucial; otherwise an evil twin can solicit credentials through a fraudulent enterprise authentication endpoint.

The function below is a labeled teaching KDF, not a drop-in implementation of 802.11. It illustrates why both peers canonicalize addresses and nonces and why a fresh nonce creates a different session key.

In [11]:
import hashlib
import hmac


def teaching_pairwise_key(pmk, ap_mac, station_mac, anonce, snonce, length=32):
    addresses = min(ap_mac, station_mac) + max(ap_mac, station_mac)
    nonces = min(anonce, snonce) + max(anonce, snonce)
    context = b"Pairwise key expansion|" + addresses + nonces
    return hmac.new(pmk, context, hashlib.sha256).digest()[:length]


pmk = bytes.fromhex("91" * 32)
ap = bytes.fromhex("001122334455")
station = bytes.fromhex("aabbccddeeff")
anonce = bytes.fromhex("10" * 32)
snonce_a = bytes.fromhex("20" * 32)
snonce_b = bytes.fromhex("21" * 32)

key_a_from_station = teaching_pairwise_key(pmk, ap, station, anonce, snonce_a)
key_a_from_ap = teaching_pairwise_key(pmk, station, ap, snonce_a, anonce)
key_b = teaching_pairwise_key(pmk, ap, station, anonce, snonce_b)

print("both peers derive the same session key:", key_a_from_station == key_a_from_ap)
print("fresh station nonce changes the key:", key_a_from_station != key_b)
print("session key A prefix:", key_a_from_station.hex()[:16])
print("session key B prefix:", key_b.hex()[:16])

both peers derive the same session key: True
fresh station nonce changes the key: True
session key A prefix: 127552667ea896e4
session key B prefix: e056199c4da248fb


#### **Rogue Access Points and Evil Twins**

A rogue AP is unauthorized infrastructure attached to a network. An **evil twin** imitates a trusted SSID to attract clients. SSID names are not authentication. Defenses include enterprise certificate validation, managed profiles, mutual authentication where appropriate, wireless monitoring, disabling automatic connection to untrusted open networks, and isolating guest access. Captive portals authenticate web access, not the radio network itself.

#### **Deauthentication and Wireless Availability**

Legacy 802.11 management frames could be spoofed to disconnect clients. Protected Management Frames authenticate selected management traffic and are required in WPA3 deployments, but radio jamming and interference remain availability problems. Link encryption also ends at the AP; the AP, upstream network, resolver, and service still require their own controls.

### **Privacy and Metadata**

#### **DNS Privacy with DoT and DoH**

DNSSEC and encrypted DNS answer different questions. DNSSEC lets a validator authenticate signed DNS data. DNS over TLS (DoT) and DNS over HTTPS (DoH) protect the channel between a client and recursive resolver from local observation and modification. The resolver still learns queries, and resolver-to-authoritative traffic may remain visible. Encrypted DNS can also concentrate metadata at one resolver, so authentication, retention, jurisdiction, and policy matter.

#### **Traffic Analysis and Fingerprinting**

TLS hides application content but ordinarily exposes source and destination IP addresses, packet sizes, timing, direction, and total volume. These features can support traffic classification or correlation. Padding, multiplexing, batching, cover traffic, relays, and larger anonymity sets can reduce leakage, each with latency and bandwidth costs.

TLS Encrypted Client Hello (ECH), standardized in [RFC 9849](https://www.rfc-editor.org/rfc/rfc9849) in 2026, encrypts the sensitive inner ClientHello, including the real server name, when configuration and negotiation succeed. ECH does not hide the destination IP, traffic shape, DNS metadata from the chosen resolver, or plaintext from the client-facing service that processes the inner hello.

![Different privacy technologies hide some metadata while shifting visibility to resolvers tunnel providers or relays.](assets/privacy-metadata-map.svg){fig-alt="Comparison of metadata visible with plain DNS and HTTP encrypted DNS and TLS ECH VPN and Tor" width="100%"}

#### **Address, Identifier, and Location Privacy**

Stable IP addresses, IPv6 interface identifiers, cookies, TLS tickets, device identifiers, and account logins can link activity across sessions. Temporary addresses reduce one linkability source but do not defeat account or application tracking. NAT changes address sharing but is not an anonymity system. Privacy engineering minimizes identifier scope and lifetime, separates roles, rotates keys and tokens deliberately, and avoids logging more data than an operational purpose requires.

#### **Anonymity Networks and Tor Overview**

Tor constructs a circuit through a guard, middle relay, and exit so that no ordinary relay should know both client and destination. Encryption is layered: each relay removes one layer and learns only adjacent hops. The exit connects to the destination, so end-to-end TLS is still required to protect application content and authenticate the service.

![Tor separates knowledge across entry middle and exit relays.](assets/tor-onion-routing.png){fig-alt="Tor client routes through three encrypted relays before reaching a destination server" width="75%"}

*Figure source: [Electronic Frontier Foundation, How Tor Works, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:How_Tor_Works_2.svg), licensed under CC BY 3.0.*

Tor is a low-latency anonymity network, not a guarantee against a powerful observer who can correlate traffic at both ends. Browser behavior, downloads, logins, malicious endpoints, and application identifiers can also defeat anonymity independently of the relay path. The [Tor specifications](https://spec.torproject.org/intro/index.html) describe relay directories, circuits, streams, and threat assumptions.

In [12]:
import math
from collections import defaultdict

# Synthetic response totals: a toy model of one feature visible to a traffic observer.
observed_bytes = {
    "news.example": 38_400,
    "mail.example": 46_200,
    "docs.example": 51_900,
    "video.example": 141_000,
}


def candidates_by_observation(values, padding_block=None):
    groups = defaultdict(list)
    for site, size in values.items():
        observed = size
        if padding_block:
            observed = math.ceil(size / padding_block) * padding_block
        groups[observed].append(site)
    return groups


print("Without padding:")
for size, candidates in sorted(candidates_by_observation(observed_bytes).items()):
    print(f"  observed {size:6d} bytes -> {candidates}")

print("\nWith coarse 64 KiB response padding:")
for size, candidates in sorted(
    candidates_by_observation(observed_bytes, padding_block=65_536).items()
):
    print(f"  observed {size:6d} bytes -> {candidates}")

Without padding:
  observed  38400 bytes -> ['news.example']
  observed  46200 bytes -> ['mail.example']
  observed  51900 bytes -> ['docs.example']
  observed 141000 bytes -> ['video.example']

With coarse 64 KiB response padding:
  observed  65536 bytes -> ['news.example', 'mail.example', 'docs.example']
  observed 196608 bytes -> ['video.example']


Padding increases ambiguity in this one-feature toy model but consumes bandwidth and does not hide timing, direction, packet count, or repeated behavior. Privacy must be evaluated against a stated observer and the complete trace, not one field.

### **Secure Network Design and Incident Response**

#### **Defense in Depth and Least Privilege**

Defense in depth creates independent barriers: authenticated access, segmented reachability, service authorization, hardened endpoints, key isolation, rate limits, backups, and monitoring. Repeating one assumption is not depth. Three IP-based ACLs can all fail when a trusted workload is compromised. Controls should reduce blast radius and preserve recovery even when one identity, zone, or dependency fails.

Least privilege applies to users, workloads, routes, keys, management APIs, logs, and time. Short-lived credentials and narrow scopes reduce the value of theft. Separate management and data planes, use change review for high-impact routing or firewall policy, and make emergency access visible and revocable.

#### **Logging, Detection, Containment, and Recovery**

An incident is managed through evidence and explicit decisions: validate the signal, establish a timeline and blast radius, contain harm, remove the cause, restore safely, and improve controls. Network evidence may include identity events, DNS and proxy logs, flow records, packet captures, endpoint telemetry, cloud control-plane changes, certificate events, and routing updates.

![Incident response moves from detection through recovery and learning while preserving synchronized evidence.](assets/incident-response-evidence-loop.svg){fig-alt="Six stage incident response loop supported by identity DNS flow endpoint and configuration evidence" width="96%"}

Clock synchronization and identifiers determine whether these streams can be joined. Collection gaps, sampling, NAT, proxying, and load balancing should be recorded. Containment changes need an owner, timestamp, intended effect, rollback condition, and verification signal; otherwise responders can mistake their own mitigation for attacker behavior.

In [13]:
from datetime import datetime, timezone

events = [
    {"time": "2026-07-20T10:01:04Z", "source": "identity", "session": "s-77", "event": "login success for alice from device d-9"},
    {"time": "2026-07-20T10:01:10Z", "source": "dns", "session": "s-77", "event": "query newly-seen.example"},
    {"time": "2026-07-20T10:01:12Z", "source": "proxy", "session": "s-77", "event": "POST /admin/export -> 200, 48 MB"},
    {"time": "2026-07-20T10:01:13Z", "source": "endpoint", "session": "s-77", "event": "unsigned process opened browser cookie store"},
    {"time": "2026-07-20T10:03:40Z", "source": "policy", "session": "s-77", "event": "session revoked and export permission removed"},
    {"time": "2026-07-20T10:04:02Z", "source": "proxy", "session": "s-77", "event": "POST /admin/export -> 401"},
    {"time": "2026-07-20T10:00:55Z", "source": "identity", "session": "s-12", "event": "normal login for bob"},
]


def parse_time(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


scoped = sorted(
    (event for event in events if event["session"] == "s-77"),
    key=lambda event: parse_time(event["time"]),
)

start = parse_time(scoped[0]["time"])
print("Correlated timeline for session s-77")
for event in scoped:
    elapsed = (parse_time(event["time"]) - start).total_seconds()
    print(f"+{elapsed:6.0f}s [{event['source']:8s}] {event['event']}")

print("\nVerification: the post-containment export attempt was denied.")

Correlated timeline for session s-77
+     0s [identity] login success for alice from device d-9
+     6s [dns     ] query newly-seen.example
+     8s [proxy   ] POST /admin/export -> 200, 48 MB
+     9s [endpoint] unsigned process opened browser cookie store
+   156s [policy  ] session revoked and export permission removed
+   178s [proxy   ] POST /admin/export -> 401

Verification: the post-containment export attempt was denied.


#### **Ethical and Legal Boundaries for Network Experiments**

Authorization defines the experiment. Use owned lab networks, local loopback, purpose-built ranges, or written scope. Specify targets, source addresses, rates, times, data handling, stop conditions, and contacts. Do not assume that a public IP, bug-bounty program, classroom account, or cloud tenant permits scanning, interception, credential testing, denial-of-service experiments, or access to other tenants.

Prefer passive inspection of your own traffic and synthetic traces. Minimize personal data, redact secrets, preserve chain of custody when evidence may be needed, and follow institutional policy and applicable law. Responsible disclosure should give maintainers enough reproducible evidence to fix the issue without publishing operational details that unnecessarily increase harm.

### **Inspecting a TLS Connection and Certificate Chain**

A TLS diagnosis should separate transport reachability, protocol negotiation, certificate path, service identity, and application response. `openssl s_client -connect host:443 -servername host -showcerts` can expose the presented chain and negotiated parameters, but the output must be interpreted carefully: presentation order is not trust, and successful encryption is not a hostname check unless the tool is asked to perform one.

The next cell creates a TLS 1.3 client and server entirely on loopback using the teaching chain generated earlier. The client trusts only the teaching root, requests `service.example`, validates the SAN and intermediate path, then reports the negotiated channel. Temporary key files are deleted automatically.

In [14]:
import socket
import ssl
import tempfile
import threading
from pathlib import Path

from cryptography.hazmat.primitives import serialization


root_pem = root_cert.public_bytes(serialization.Encoding.PEM)
chain_pem = (
    leaf_cert.public_bytes(serialization.Encoding.PEM)
    + intermediate_cert.public_bytes(serialization.Encoding.PEM)
)
leaf_key_pem = leaf_key.private_bytes(
    serialization.Encoding.PEM,
    serialization.PrivateFormat.PKCS8,
    serialization.NoEncryption(),
)

with tempfile.TemporaryDirectory() as temp_dir:
    temp = Path(temp_dir)
    root_file = temp / "root.pem"
    chain_file = temp / "chain.pem"
    key_file = temp / "leaf-key.pem"
    root_file.write_bytes(root_pem)
    chain_file.write_bytes(chain_pem)
    key_file.write_bytes(leaf_key_pem)

    server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    server_context.minimum_version = ssl.TLSVersion.TLSv1_3
    server_context.load_cert_chain(chain_file, key_file)

    client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    client_context.minimum_version = ssl.TLSVersion.TLSv1_3
    client_context.load_verify_locations(cafile=root_file)
    client_context.check_hostname = True

    listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    listener.bind(("127.0.0.1", 0))
    listener.listen(1)

    def serve_once():
        connection, _ = listener.accept()
        with connection:
            with server_context.wrap_socket(connection, server_side=True) as tls_socket:
                request = tls_socket.recv(16)
                tls_socket.sendall(b"secure:" + request)

    thread = threading.Thread(target=serve_once, daemon=True)
    thread.start()

    with socket.create_connection(listener.getsockname()) as raw_socket:
        with client_context.wrap_socket(
            raw_socket, server_hostname="service.example"
        ) as tls_socket:
            peer = x509.load_der_x509_certificate(tls_socket.getpeercert(binary_form=True))
            tls_socket.sendall(b"hello")
            reply = tls_socket.recv(32)
            print("TLS version:", tls_socket.version())
            print("cipher suite:", tls_socket.cipher()[0])
            print("validated peer subject:", peer.subject.rfc4514_string())
            print("application reply:", reply.decode())

    thread.join(timeout=5)
    listener.close()

TLS version: TLSv1.3
cipher suite: TLS_AES_256_GCM_SHA384
validated peer subject: CN=service.example
application reply: secure:hello


The client would abort if the root were absent, the intermediate signature failed, the leaf were expired, or `server_hostname` did not match a SAN. Those failures answer different questions and should remain distinct in logs. Disabling verification to “make TLS work” removes the authentication property that prevents active interception.

### **Threat-Modeling the End-to-End HTTPS Path**

The complete path can now be reviewed as a sequence of trust decisions:

| Phase | Important asset or claim | Representative threat | Primary controls | Residual exposure |
|---|---|---|---|---|
| join access network | local next-hop and resolver configuration | rogue AP, ARP/RA manipulation | WPA3/802.1X, PMF, switch controls, end-to-end TLS | radio timing, AP trust |
| resolve name | service locator | observation, injection, malicious resolver | DoT/DoH channel, DNSSEC validation where deployed | resolver sees query; IP remains visible |
| select Internet path | reach intended prefix | hijack or route leak | RPKI ROV, filters, monitoring, resilient routing | origin-only validation, path metadata |
| establish transport | return-path reachability and capacity | spoofing, replay, state exhaustion | cookies/tokens, anti-amplification, rate and state limits | endpoint IP and traffic shape |
| authenticate TLS | key authorized for service name | MITM, downgrade, misissuance | TLS 1.3 transcript, PKIX path, SAN match, CT | trusted CA/endpoint compromise |
| process HTTP | authorized application action | token theft, replay, object bypass | secure cookie, CSRF defense, per-action authorization, idempotency | server and browser see plaintext |
| operate and recover | availability and evidence | DDoS, control-plane abuse, hidden persistence | layered capacity, least privilege, logs, response and recovery | correlated metadata and human error |

A control matrix makes uncovered goals visible. It should include residual risks even when every planned control is enabled.

In [15]:
controls = {
    "authenticated_wifi": True,
    "encrypted_dns": True,
    "dnssec_validation": True,
    "rpki_rov": True,
    "tls13_transcript_authentication": True,
    "hostname_validation": True,
    "per_action_authorization": True,
    "rate_limits_and_capacity": True,
    "endpoint_hardening": True,
}


def residual_findings(enabled):
    findings = []
    if not enabled["authenticated_wifi"]:
        findings.append("local access network can be impersonated")
    if not enabled["encrypted_dns"]:
        findings.append("local observers see DNS names")
    if not enabled["dnssec_validation"]:
        findings.append("encrypted resolver channel does not prove authoritative DNS data")
    if not enabled["rpki_rov"]:
        findings.append("unauthorized route origins are not filtered by RPKI state")
    if not enabled["tls13_transcript_authentication"]:
        findings.append("handshake key shares and negotiated transcript are not authenticated")
    if not enabled["hostname_validation"]:
        findings.append("a valid certificate key may belong to a different service identity")
    if not enabled["per_action_authorization"]:
        findings.append("a valid session may access unauthorized objects")
    if not enabled["rate_limits_and_capacity"]:
        findings.append("valid-looking requests can exhaust constrained resources")
    if not enabled["endpoint_hardening"]:
        findings.append("malware can read plaintext and credentials at the endpoint")

    # These remain even with the planned controls enabled.
    findings.extend([
        "destination IP, timing, and volume remain observable on the path",
        "trusted endpoints and selected service providers can see plaintext or metadata",
    ])
    return findings


for label, changed in [
    ("all planned controls", {}),
    ("hostname verification disabled", {"hostname_validation": False}),
    ("application authorization missing", {"per_action_authorization": False}),
]:
    scenario = controls | changed
    print(f"\n{label}:")
    for finding in residual_findings(scenario):
        print(" -", finding)


all planned controls:
 - destination IP, timing, and volume remain observable on the path
 - trusted endpoints and selected service providers can see plaintext or metadata

hostname verification disabled:
 - a valid certificate key may belong to a different service identity
 - destination IP, timing, and volume remain observable on the path
 - trusted endpoints and selected service providers can see plaintext or metadata

application authorization missing:
 - a valid session may access unauthorized objects
 - destination IP, timing, and volume remain observable on the path
 - trusted endpoints and selected service providers can see plaintext or metadata


The exercise reveals why a green lock icon is only one part of the system. Even a fully protected channel leaves endpoint trust, application authorization, availability, routing policy, and metadata privacy to other mechanisms.

### **Summary**

- Security goals must be attached to assets, identities, trust boundaries, and explicit attacker capabilities.
- Hashes, MACs, signatures, AEAD, key exchange, certificates, and replay windows provide different properties and must be composed with context.
- PKIX validates a path from an independently chosen service identity to a local trust anchor; signatures alone are not identity.
- TLS 1.3 authenticates a transcript and derives directional traffic keys, but 0-RTT needs application replay defenses and TLS cannot protect a compromised endpoint.
- IPsec and VPNs protect selected network paths while adding trust, routing, MTU, and key-management responsibilities.
- Firewalls, proxies, segmentation, identity-aware policy, endpoint controls, and monitoring provide independent barriers when designed around least privilege.
- DNS, transport, routing, Wi-Fi, and application attacks violate different assumptions; defenses must be placed before the resource they protect.
- RPKI Route Origin Validation authenticates origin authorization, not the entire AS path.
- Encrypted DNS, ECH, VPNs, and Tor change which observer sees metadata; none makes all traffic information disappear.
- Incident response depends on synchronized evidence, reversible containment, verified recovery, and lessons that change the system.

Chapter 9 turns these principles into operational evidence: packet captures, active and passive measurements, telemetry, troubleshooting, safe experiments, SLOs, and incident-quality timelines.